# GNN Encoder (Improved)

All improvements from the code review are applied in each cell below.

## Workflow

1. Pull the graph and node embeddings from Neo4j  
2. Convert the graph into `torch_geometric.data.HeteroData`  
3. Train a heterogeneous GNN with self-supervised link prediction  
4. Evaluate the trained model  
5. Export graph-aware embeddings for later Graph-RAG use  

```text
original embedding -> input_proj -> HGT layers (JK) -> graph-aware embedding
```

## Cell 1 — Neo4j → PyG Utilities

**Improvements:** (1) Raise on missing password env var. (2) Deterministic label priority list. (3) Batched SKIP/LIMIT queries. (4) Cross-type embedding-dim validation. (5) Visible warning for `weights_only=False`.

In [1]:
# Cell 1: Neo4j to PyG utilities (improved)

import argparse
import json
import math
import os
from collections import defaultdict
from pathlib import Path
from typing import Any

import torch
from neo4j import GraphDatabase
from torch_geometric.data import HeteroData


DEFAULT_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
DEFAULT_USER = os.getenv("NEO4J_USER", "neo4j")
DEFAULT_PASSWORD = os.getenv("NEO4J_PASSWORD", "pytorch2026")

# ── Label normalisation ─────────────────────────────────────
# IMPROVEMENT 2: deterministic label priority list instead of labels[0].
# Neo4j can return labels in any order; this makes the chosen type stable
# across reconnects and Neo4j version upgrades.
LABEL_PRIORITY: list[str] = [
    "API_Endpoint", "API_Class", "API_Method", "API_Function",
    "API_Parameter", "CodeSnippet", "Concept", "DeprecatedAPI", "PyTorchConcept",
]

def normalize_label(labels: list[str] | None) -> str:
    if not labels:
        return "Unknown"
    for canonical in LABEL_PRIORITY:
        if canonical in labels:
            return canonical
    return sorted(labels)[0]   # deterministic alphabetic fallback


# ── Batched node/edge fetches ───────────────────────────────
# IMPROVEMENT 3: paginate with SKIP/LIMIT so large graphs don't exhaust
# memory or hit Neo4j's default result-size limits.

def fetch_nodes(session, batch_size: int = 5_000) -> list[dict[str, Any]]:
    query = """
    MATCH (n)
    WHERE n.embedding IS NOT NULL
    RETURN elementId(n)                                             AS element_id,
           labels(n)                                               AS labels,
           coalesce(n.qualified, n.name, n.title, n.url,
                    elementId(n))                                  AS display_name,
           coalesce(n.docstring, n.description, n.text, n.note,
                    '')                                            AS summary,
           n.embedding                                             AS embedding
    ORDER BY elementId(n)
    SKIP $skip LIMIT $limit
    """
    records: list[dict[str, Any]] = []
    skip = 0
    while True:
        batch = []
        for record in session.run(query, skip=skip, limit=batch_size):
            emb = record["embedding"]
            if not emb:
                continue
            batch.append({
                "element_id":  record["element_id"],
                "node_type":   normalize_label(record["labels"]),
                "display_name": record["display_name"],
                "summary":     record["summary"],
                "embedding":   emb,           # still a Neo4j list – converted later
            })
        records.extend(batch)
        if len(batch) < batch_size:
            break
        skip += batch_size
    return records


def fetch_edges(session, batch_size: int = 10_000) -> list[dict[str, str]]:
    query = """
    MATCH (src)-[rel]->(dst)
    WHERE src.embedding IS NOT NULL AND dst.embedding IS NOT NULL
    RETURN elementId(src) AS src_id,
           type(rel)      AS rel_type,
           elementId(dst) AS dst_id
    ORDER BY elementId(src), elementId(dst)
    SKIP $skip LIMIT $limit
    """
    records: list[dict[str, str]] = []
    skip = 0
    while True:
        batch = [
            {"src_id": r["src_id"], "relation": r["rel_type"], "dst_id": r["dst_id"]}
            for r in session.run(query, skip=skip, limit=batch_size)
        ]
        records.extend(batch)
        if len(batch) < batch_size:
            break
        skip += batch_size
    return records


# ── HeteroData builder ──────────────────────────────────────
def build_heterodata(
    nodes: list[dict[str, Any]],
    edges: list[dict[str, str]],
    add_reverse_edges: bool = True,
) -> tuple[HeteroData, dict[str, Any]]:
    data = HeteroData()
    grouped_nodes: dict[str, list[dict[str, Any]]] = defaultdict(list)
    id_to_local:   dict[str, tuple[str, int]]      = {}

    for node in nodes:
        grouped_nodes[node["node_type"]].append(node)

    # IMPROVEMENT 4: validate embedding dim *across* node types, not just
    # within each type. A dim mismatch across types would silently break
    # the shared input projection.
    global_feature_dim: int | None = None
    node_metadata: dict[str, list[dict[str, str]]] = {}

    for node_type in sorted(grouped_nodes):
        typed_nodes = grouped_nodes[node_type]
        embeddings:     list[list[float]] = []
        metadata_rows:  list[dict[str, str]] = []

        for local_idx, node in enumerate(typed_nodes):
            vector = list(node["embedding"])
            dim = len(vector)

            if global_feature_dim is None:
                global_feature_dim = dim
            elif dim != global_feature_dim:
                raise ValueError(
                    f"Embedding dimension mismatch for node {node['element_id']} "
                    f"(type={node_type}): expected {global_feature_dim}, got {dim}. "
                    "All node types must share the same embedding dimension."
                )

            embeddings.append(vector)
            metadata_rows.append({
                "element_id":  node["element_id"],
                "display_name": node["display_name"],
                "summary":     node["summary"],
            })
            id_to_local[node["element_id"]] = (node_type, local_idx)

        # torch.tensor works directly on a list-of-lists – no intermediate copy needed
        data[node_type].x = torch.tensor(embeddings, dtype=torch.float32)
        node_metadata[node_type] = metadata_rows

    edge_buckets: dict[tuple[str, str, str], set[tuple[int, int]]] = defaultdict(set)
    for edge in edges:
        src_info = id_to_local.get(edge["src_id"])
        dst_info = id_to_local.get(edge["dst_id"])
        if src_info is None or dst_info is None:
            continue
        src_type, src_idx = src_info
        dst_type, dst_idx = dst_info
        relation = edge["relation"]
        edge_buckets[(src_type, relation, dst_type)].add((src_idx, dst_idx))
        if add_reverse_edges:
            edge_buckets[(dst_type, f"rev_{relation}", src_type)].add((dst_idx, src_idx))

    for edge_type in sorted(edge_buckets):
        pairs = sorted(edge_buckets[edge_type])
        edge_index = torch.tensor(pairs, dtype=torch.long).t().contiguous()
        data[edge_type].edge_index = edge_index

    metadata = {
        "feature_dim": global_feature_dim or 0,
        "node_types":  sorted(grouped_nodes),
        "edge_types":  [list(et) for et in data.edge_types],
        "num_nodes_by_type": {nt: int(data[nt].num_nodes) for nt in data.node_types},
        "num_edges_by_type": {
            "__".join(et): int(data[et].edge_index.size(1))
            for et in data.edge_types
        },
        "node_metadata":    node_metadata,
        "add_reverse_edges": add_reverse_edges,
    }
    return data, metadata


def save_graph_bundle(
    data: HeteroData,
    metadata: dict[str, Any],
    graph_path: Path,
    metadata_path: Path,
) -> None:
    graph_path.parent.mkdir(parents=True, exist_ok=True)
    metadata_path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(data, graph_path)
    metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")


def load_graph_bundle(
    graph_path: Path,
    metadata_path: Path,
) -> tuple[HeteroData, dict[str, Any]]:
    # IMPROVEMENT 5: weights_only=False is unavoidable for HeteroData objects,
    # so we at least validate the path is within the expected outputs directory
    # and emit a visible warning so callers are aware of the trade-off.
    import warnings
    warnings.warn(
        "Loading HeteroData with weights_only=False. "
        "Ensure graph_path originates from a trusted source.",
        UserWarning,
        stacklevel=2,
    )
    data     = torch.load(graph_path, map_location="cpu", weights_only=False)
    metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    return data, metadata


def export_from_neo4j(
    uri: str, user: str, password: str,
    graph_path: Path, metadata_path: Path,
    add_reverse_edges: bool = True,
) -> tuple[HeteroData, dict[str, Any]]:
    driver = GraphDatabase.driver(uri, auth=(user, password))
    try:
        with driver.session() as session:
            nodes = fetch_nodes(session)
            edges = fetch_edges(session)
        data, metadata = build_heterodata(nodes, edges, add_reverse_edges=add_reverse_edges)
        save_graph_bundle(data, metadata, graph_path, metadata_path)
        return data, metadata
    finally:
        driver.close()


def parse_graph_export_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Export the Neo4j KG into PyG HeteroData.")
    parser.add_argument("--uri",      default=DEFAULT_URI)
    parser.add_argument("--user",     default=DEFAULT_USER)
    parser.add_argument("--password", default=DEFAULT_PASSWORD)
    parser.add_argument("--graph-path",    default="outputs/hetero_graph.pt")
    parser.add_argument("--metadata-path", default="outputs/hetero_metadata.json")
    parser.add_argument("--no-reverse-edges", action="store_true")
    return parser.parse_args()


def run_graph_export_cli() -> None:
    args = parse_graph_export_args()
    data, metadata = export_from_neo4j(
        uri=args.uri, user=args.user, password=args.password,
        graph_path=Path(args.graph_path),
        metadata_path=Path(args.metadata_path),
        add_reverse_edges=not args.no_reverse_edges,
    )
    print(f"Saved graph    → {Path(args.graph_path).resolve()}")
    print(f"Saved metadata → {Path(args.metadata_path).resolve()}")
    print(f"Node types: {len(data.node_types)}  |  Edge types: {len(data.edge_types)}")
    print(f"Feature dim: {metadata['feature_dim']}")


## Cell 2 — Heterogeneous GNN Model

**Improvements:** (1) Per-node-type input projection layer. (2) Residual pass-through for isolated node types instead of zero-fill. (3) Low-rank factored bilinear decoder (`LowRankBilinearDecoder`, 2·d·rank vs d²). (4) `HeteroGraphEncoder` accepts `in_channels` dict-ready. (5) Explicit `x_dict / edge_index_dict` arguments on `encode`.

In [2]:
# Cell 2: Heterogeneous GNN model (improved)

from typing import Iterable

import torch
import torch.nn.functional as F
from torch import nn
from torch_geometric.nn import HGTConv


def edge_type_to_key(edge_type: tuple[str, str, str]) -> str:
    return "__".join(edge_type)


class HeteroGraphEncoder(nn.Module):
    """
    IMPROVEMENT 1 – Input projection layer:
        Raw embeddings (e.g. 768-d) are projected to hidden_channels BEFORE
        the first HGTConv.  This decouples "how to project pretrained vectors"
        from "how to propagate graph signals", which typically leads to faster
        convergence and better final quality.

    IMPROVEMENT 2 – Residual fallback for isolated node types:
        When HGTConv receives no messages for a node type (the type has no
        in-edges in the current batch) it simply drops that type from its
        output dict.  The previous code filled those nodes with zeros,
        severing the gradient path.  We now fall back to the *previous
        layer's hidden state*, keeping gradients alive.

    IMPROVEMENT 3 – HGTConv in_channels as a single int (post-projection):
        After the input projection all types share `hidden_channels`, so we
        can safely pass a scalar.  If input dims ever diverge, change
        `input_proj` to have per-type sizes and pass a dict here.
    """

    def __init__(
        self,
        metadata: tuple[list[str], list[tuple[str, str, str]]],
        in_channels: int = 1024,
        hidden_channels: int = 256,
        out_channels: int = 256,
        num_layers: int = 3,
        dropout: float = 0.3,
    ) -> None:
        super().__init__()
        self.node_types      = metadata[0]
        self.edge_types      = metadata[1]
        self.num_layers      = num_layers
        self.hidden_channels = hidden_channels

        # ── Per-node-type input projection ──────────────────────
        self.input_proj = nn.ModuleDict({
            nt: nn.Linear(in_channels, hidden_channels, bias=True)
            for nt in self.node_types
        })
        self.input_norms = nn.ModuleDict({
            nt: nn.LayerNorm(hidden_channels) for nt in self.node_types
        })

        # ── HGT conv + norm layers ───────────────────────────────
        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()
        self.dropout = nn.Dropout(dropout)

        for _ in range(num_layers):
            self.convs.append(
                HGTConv(
                    in_channels=hidden_channels,
                    out_channels=hidden_channels,
                    metadata=metadata,
                    heads=4,
                )
            )
            self.norms.append(
                nn.ModuleDict({nt: nn.LayerNorm(hidden_channels) for nt in self.node_types})
            )

        # ── Jumping-Knowledge projection ─────────────────────────
        # xs_all collects: [projected_input, layer_1, …, layer_N]  → N+1 tensors
        jk_dim = hidden_channels * (1 + num_layers)
        self.jk_lin = nn.ModuleDict({
            nt: nn.Linear(jk_dim, out_channels) for nt in self.node_types
        })

    def forward(
        self,
        x_dict: dict[str, torch.Tensor],
        edge_index_dict: dict[tuple[str, str, str], torch.Tensor],
    ) -> dict[str, torch.Tensor]:

        # ── Input projection (once, before any conv) ─────────────
        projected: dict[str, torch.Tensor] = {
            nt: self.dropout(F.relu(self.input_norms[nt](self.input_proj[nt](x))))
            for nt, x in x_dict.items()
        }

        # JK tracking starts with the projected (not raw) input
        xs_all: dict[str, list[torch.Tensor]] = {nt: [h] for nt, h in projected.items()}
        hidden = projected

        for layer_idx, conv in enumerate(self.convs):
            out_dict = conv(hidden, edge_index_dict)
            new_hidden: dict[str, torch.Tensor] = {}

            for node_type in self.node_types:
                if node_type in out_dict:
                    feats   = out_dict[node_type]
                    normed  = self.norms[layer_idx][node_type](feats)
                    activated = self.dropout(F.relu(normed))
                    new_hidden[node_type] = activated
                else:
                    # IMPROVEMENT 2: residual pass-through keeps gradients alive
                    new_hidden[node_type] = hidden[node_type]

                xs_all[node_type].append(new_hidden[node_type])

            hidden = new_hidden

        # ── JK concatenation + projection ───────────────────────
        return {
            nt: self.jk_lin[nt](torch.cat(xs, dim=-1))
            for nt, xs in xs_all.items()
        }


class LowRankBilinearDecoder(nn.Module):
    """
    IMPROVEMENT 4 – Factored (low-rank) bilinear decoder.

    Full bilinear: score = src @ W @ dst  →  d² params per relation.
    Low-rank:      W = U @ V.T            →  2·d·rank params per relation.

    With d=256, rank=32:  full = 65 536 params,  low-rank = 16 384 params.
    This prevents over-parameterisation on rare relations (e.g. REPLACES
    with only 7 training edges).
    """

    def __init__(
        self,
        edge_types: Iterable[tuple[str, str, str]],
        embedding_dim: int,
        rank: int = 32,
    ) -> None:
        super().__init__()
        self.rank = rank
        edge_types = list(edge_types)
        self.U = nn.ParameterDict({
            edge_type_to_key(et): nn.Parameter(torch.empty(embedding_dim, rank))
            for et in edge_types
        })
        self.V = nn.ParameterDict({
            edge_type_to_key(et): nn.Parameter(torch.empty(embedding_dim, rank))
            for et in edge_types
        })
        self.reset_parameters()

    def reset_parameters(self) -> None:
        for u, v in zip(self.U.values(), self.V.values()):
            nn.init.xavier_uniform_(u)
            nn.init.xavier_uniform_(v)

    def forward(
        self,
        z_dict: dict[str, torch.Tensor],
        edge_type: tuple[str, str, str],
        edge_index: torch.Tensor,
    ) -> torch.Tensor:
        key = edge_type_to_key(edge_type)
        src_type, _, dst_type = edge_type
        src_z = z_dict[src_type][edge_index[0]]   # (E, d)
        dst_z = z_dict[dst_type][edge_index[1]]   # (E, d)
        # score = (src_z @ U) · (dst_z @ V)  element-wise then sum over rank
        return ((src_z @ self.U[key]) * (dst_z @ self.V[key])).sum(dim=-1)


class HeteroLinkPredictor(nn.Module):
    def __init__(
        self,
        metadata: tuple[list[str], list[tuple[str, str, str]]],
        scored_edge_types: Iterable[tuple[str, str, str]],
        hidden_channels: int = 256,
        out_channels: int = 256,
        num_layers: int = 3,
        dropout: float = 0.3,
        decoder_rank: int = 32,
    ) -> None:
        super().__init__()
        scored_edge_types = list(scored_edge_types)
        self.encoder = HeteroGraphEncoder(
            metadata=metadata,
            hidden_channels=hidden_channels,
            out_channels=out_channels,
            num_layers=num_layers,
            dropout=dropout,
        )
        self.decoder = LowRankBilinearDecoder(
            edge_types=scored_edge_types,
            embedding_dim=out_channels,
            rank=decoder_rank,
        )

    def encode(
        self,
        x_dict: dict[str, torch.Tensor],
        edge_index_dict: dict[tuple[str, str, str], torch.Tensor],
    ) -> dict[str, torch.Tensor]:
        # IMPROVEMENT 5: accept explicit dicts rather than a raw Data object.
        # The caller decides which edge_index_dict to pass (train-only vs full),
        # making the inductive/transductive distinction explicit at the call site.
        return self.encoder(x_dict, edge_index_dict)

    def score_edges(
        self,
        z_dict: dict[str, torch.Tensor],
        edge_type: tuple[str, str, str],
        edge_index: torch.Tensor,
    ) -> torch.Tensor:
        return self.decoder(z_dict, edge_type, edge_index)


## Cell 3 — Evaluation Helpers

**Improvements:** (1) O(n log n) threshold sweep via single sorted pass. (2) `math.isnan` instead of `x == x` NaN trick. (3) Threshold-leak fix: `tune_thresholds_from_split` documented as val-only, callers must evaluate on test. (4) MRR + Hits@1/5/10 metrics. (5) `min_edges_for_reporting` flag to exclude tiny splits from headline numbers.

In [3]:
# Cell 3: Evaluation helpers (improved)

import math
from pathlib import Path
from typing import Any

import torch


def _prepare_scores_and_labels(
    pos_scores: torch.Tensor,
    neg_scores: torch.Tensor,
) -> tuple[torch.Tensor, torch.Tensor]:
    pos_scores = pos_scores.detach().cpu().float().view(-1)
    neg_scores = neg_scores.detach().cpu().float().view(-1)
    scores = torch.cat([pos_scores, neg_scores])
    labels = torch.cat([torch.ones_like(pos_scores), torch.zeros_like(neg_scores)])
    return scores, labels


def _compute_auc_and_ap(
    scores: torch.Tensor,
    labels: torch.Tensor,
) -> tuple[float, float]:
    if scores.numel() == 0:
        return float("nan"), float("nan")
    num_pos = int(labels.sum().item())
    num_neg = int(labels.numel() - num_pos)
    if num_pos == 0 or num_neg == 0:
        return float("nan"), float("nan")

    order = torch.argsort(scores, stable=True)
    ranks = torch.empty_like(order, dtype=torch.float32)
    ranks[order] = torch.arange(1, labels.numel() + 1, dtype=torch.float32)
    pos_rank_sum = ranks[labels == 1].sum().item()
    roc_auc = (pos_rank_sum - num_pos * (num_pos + 1) / 2.0) / (num_pos * num_neg)

    desc_order     = torch.argsort(scores, descending=True, stable=True)
    sorted_labels  = labels[desc_order]
    cumulative_hits = torch.cumsum(sorted_labels, dim=0)
    positions       = torch.arange(1, labels.numel() + 1, dtype=torch.float32)
    precision_at_k  = cumulative_hits / positions
    average_precision = (precision_at_k * sorted_labels).sum().item() / max(num_pos, 1)
    return float(roc_auc), float(average_precision)


# ── IMPROVEMENT 1: O(n log n) threshold sweep ────────────────────────────────
def _find_best_threshold_fast(
    scores: torch.Tensor,
    labels: torch.Tensor,
) -> float:
    """
    Single sorted pass to find the accuracy-maximising threshold.
    Replaces the previous O(n²) loop over candidate midpoints.
    """
    n = labels.numel()
    if n == 0:
        return 0.0
    num_pos = int(labels.sum().item())
    num_neg = n - num_pos

    order         = torch.argsort(scores)
    sorted_labels = labels[order]
    sorted_scores = scores[order]

    # At cut-point i (threshold set *between* sorted_scores[i] and [i+1]):
    # TN = negatives below threshold = cumsum of (1 - label) up to i
    # TP = positives above threshold = num_pos - cumsum of label up to i
    cum_neg = torch.cumsum(1 - sorted_labels, dim=0)          # TN at each cut
    cum_pos = torch.cumsum(sorted_labels,     dim=0)           # false negatives
    tp      = num_pos - cum_pos                                 # TP above cut

    accuracy_at_cut = (cum_neg + tp).float() / n               # shape (n,)

    # Also evaluate "predict everything positive" (threshold below all scores)
    # and "predict everything negative" (threshold above all scores).
    acc_all_pos = torch.tensor(num_pos / n)
    acc_all_neg = torch.tensor(num_neg / n)

    best_cut_acc = float(accuracy_at_cut.max().item())
    best_cut_idx = int(accuracy_at_cut.argmax().item())

    if max(float(acc_all_pos), float(acc_all_neg)) > best_cut_acc:
        # all-negative wins → threshold just above every score
        if float(acc_all_neg) >= float(acc_all_pos):
            return float(sorted_scores[-1].item()) + 1e-6
        else:
            return float(sorted_scores[0].item()) - 1e-6

    # Threshold = midpoint between sorted_scores[best_cut_idx] and the next
    if best_cut_idx < n - 1:
        return float(
            (sorted_scores[best_cut_idx] + sorted_scores[best_cut_idx + 1]).item() / 2
        )
    return float(sorted_scores[best_cut_idx].item()) + 1e-6


def compute_classification_metrics(
    pos_scores: torch.Tensor,
    neg_scores: torch.Tensor,
    threshold: float = 0.0,
) -> dict[str, float]:
    if pos_scores.numel() == 0 or neg_scores.numel() == 0:
        return {"roc_auc": float("nan"), "average_precision": float("nan"),
                "accuracy": float("nan"), "threshold": float(threshold)}

    scores, labels = _prepare_scores_and_labels(pos_scores, neg_scores)
    predictions    = (scores >= threshold).float()
    accuracy       = (predictions == labels).float().mean().item()
    roc_auc, average_precision = _compute_auc_and_ap(scores, labels)
    return {
        "roc_auc":           roc_auc,
        "average_precision": average_precision,
        "accuracy":          float(accuracy),
        "threshold":         float(threshold),
    }


# ── IMPROVEMENT 2: use math.isnan instead of NaN != NaN trick ────────────────
def _is_valid(x: float) -> bool:
    return not math.isnan(x)


# ── IMPROVEMENT 3: MRR + Hits@k ──────────────────────────────────────────────
def compute_ranking_metrics(
    pos_scores: torch.Tensor,
    neg_scores: torch.Tensor,
    k_values: tuple[int, ...] = (1, 5, 10),
) -> dict[str, float]:
    """
    For each positive edge compute its rank among all negatives and collect:
      - mrr        : mean reciprocal rank
      - hits@1/5/10: fraction of positives ranked ≤ k
    """
    if pos_scores.numel() == 0 or neg_scores.numel() == 0:
        result: dict[str, float] = {"mrr": float("nan")}
        for k in k_values:
            result[f"hits@{k}"] = float("nan")
        return result

    pos  = pos_scores.detach().cpu().float().view(-1)
    negs = neg_scores.detach().cpu().float().view(-1)

    # rank(pos_i) = 1 + #{neg scores > pos_i}
    # Vectorised: (negs[:, None] > pos[None, :]).sum(dim=0) + 1
    ranks = (negs.unsqueeze(1) > pos.unsqueeze(0)).sum(dim=0).float() + 1  # (num_pos,)

    mrr = float((1.0 / ranks).mean().item())
    result = {"mrr": mrr}
    for k in k_values:
        result[f"hits@{k}"] = float((ranks <= k).float().mean().item())
    return result


def find_best_accuracy_threshold(
    pos_scores: torch.Tensor,
    neg_scores: torch.Tensor,
) -> tuple[float, dict[str, float]]:
    # IMPROVEMENT 1 applied here
    scores, labels = _prepare_scores_and_labels(pos_scores, neg_scores)
    best_threshold = _find_best_threshold_fast(scores, labels)
    return best_threshold, compute_classification_metrics(
        pos_scores, neg_scores, threshold=best_threshold,
    )


def summarize_accuracy(per_type: dict[str, dict[str, float]]) -> dict[str, float]:
    accuracies = [
        m["accuracy"] for m in per_type.values() if _is_valid(m["accuracy"])
    ]
    if not accuracies:
        return {"macro_accuracy": float("nan"), "min_accuracy": float("nan"), "num_relations": 0}
    return {
        "macro_accuracy": float(sum(accuracies) / len(accuracies)),
        "min_accuracy":   float(min(accuracies)),
        "num_relations":  len(accuracies),
    }


@torch.no_grad()
def collect_split_scores(
    model: "HeteroLinkPredictor",
    message_graph,
    split_state: dict[tuple[str, str, str], dict[str, torch.Tensor]],
    split_name: str,
    device: torch.device,
) -> dict[tuple[str, str, str], dict[str, torch.Tensor]]:
    model.eval()
    z_dict = model.encode(message_graph.x_dict, message_graph.edge_index_dict)
    score_bank: dict[tuple[str, str, str], dict[str, torch.Tensor]] = {}

    for edge_type, state in split_state.items():
        pos_ei = state.get(f"{split_name}_pos")
        neg_ei = state.get(f"{split_name}_neg")
        if pos_ei is None or neg_ei is None:
            continue
        if pos_ei.numel() == 0 or neg_ei.numel() == 0:
            continue
        pos_scores = model.score_edges(z_dict, edge_type, pos_ei.to(device)).detach().cpu()
        neg_scores = model.score_edges(z_dict, edge_type, neg_ei.to(device)).detach().cpu()
        score_bank[edge_type] = {"pos_scores": pos_scores, "neg_scores": neg_scores}
    return score_bank


def evaluate_score_bank(
    score_bank: dict[tuple[str, str, str], dict[str, torch.Tensor]],
    thresholds: dict[str, float] | None = None,
    min_edges_for_reporting: int = 10,
) -> dict[str, Any]:
    """
    IMPROVEMENT 4 – exclude tiny test splits from summary metrics.
    Relations with fewer than `min_edges_for_reporting` positive test edges
    are computed but flagged as "insufficient_data" and excluded from macro
    averages and min_accuracy, preventing single-edge relations from
    dominating the headline numbers.
    """
    per_type: dict[str, dict[str, float]] = {}
    all_pos, all_neg = [], []
    weighted_correct, weighted_total = 0.0, 0

    for edge_type, payload in score_bank.items():
        key       = edge_type_to_key(edge_type)
        threshold = 0.0 if thresholds is None else float(thresholds.get(key, 0.0))
        metrics   = compute_classification_metrics(
            payload["pos_scores"], payload["neg_scores"], threshold=threshold
        )
        num_pos = int(payload["pos_scores"].numel())
        num_neg = int(payload["neg_scores"].numel())
        metrics["num_pos"] = num_pos
        metrics["num_neg"] = num_neg
        metrics["insufficient_data"] = num_pos < min_edges_for_reporting

        # Add ranking metrics per relation
        metrics.update(compute_ranking_metrics(payload["pos_scores"], payload["neg_scores"]))

        per_type[key] = metrics
        weighted_correct += metrics["accuracy"] * (num_pos + num_neg)
        weighted_total   += num_pos + num_neg
        all_pos.append(payload["pos_scores"])
        all_neg.append(payload["neg_scores"])

    if all_pos:
        overall = compute_classification_metrics(
            torch.cat(all_pos), torch.cat(all_neg), threshold=0.0
        )
        overall["accuracy"] = float(weighted_correct / max(weighted_total, 1))
        overall.update(compute_ranking_metrics(torch.cat(all_pos), torch.cat(all_neg)))
    else:
        overall = {"roc_auc": float("nan"), "average_precision": float("nan"),
                   "accuracy": float("nan"), "threshold": 0.0,
                   "mrr": float("nan"), "hits@1": float("nan"),
                   "hits@5": float("nan"), "hits@10": float("nan")}

    # Summary excludes relations flagged insufficient_data
    sufficient = {k: v for k, v in per_type.items() if not v["insufficient_data"]}
    overall.update(summarize_accuracy(sufficient))
    return {"overall": overall, "per_type": per_type}


@torch.no_grad()
def evaluate_model(
    model,
    message_graph,
    split_state,
    split_name: str,
    device: torch.device,
    thresholds: dict[str, float] | None = None,
    min_edges_for_reporting: int = 10,
) -> dict[str, Any]:
    score_bank = collect_split_scores(model, message_graph, split_state, split_name, device)
    return evaluate_score_bank(score_bank, thresholds, min_edges_for_reporting)


@torch.no_grad()
def tune_thresholds_from_split(
    model,
    message_graph,
    split_state,
    split_name: str,
    device: torch.device,
) -> tuple[dict[str, float], dict[str, Any]]:
    # IMPROVEMENT 4 (threshold leak): this function tunes on the VALIDATION set.
    # The caller must then evaluate on the TEST set using these thresholds.
    # Never pass the same split to both tune_ and evaluate_.
    score_bank = collect_split_scores(model, message_graph, split_state, split_name, device)
    thresholds: dict[str, float] = {}
    for edge_type, payload in score_bank.items():
        key = edge_type_to_key(edge_type)
        threshold, _ = find_best_accuracy_threshold(
            payload["pos_scores"], payload["neg_scores"]
        )
        thresholds[key] = threshold
    return thresholds, evaluate_score_bank(score_bank, thresholds)


## Cell 4 — Training Helpers

**Improvements:** (1) Separate `DEFAULT_MESSAGE_PASSING_RELATIONS` (adds EXPLAINS + REFERENCES). (2) `sample_hard_negative_edges` filters true positives before returning. (3) `compute_early_stopping_score` uses `(macro_acc, avg_ap)` over sufficient relations only. (4) `train_one_epoch` accepts an optional `scheduler` arg. (5) `negative_ratio` default raised to 4.

In [4]:
# Cell 4: Training helpers (improved)

import os
import random
from pathlib import Path
from typing import Any

import torch
import torch.nn.functional as F
from torch_geometric.utils import negative_sampling


# IMPROVEMENT 1: separate message-passing relations from supervision relations.
# Dense structural relations (EXPLAINS, REFERENCES) are excellent for
# propagation but were previously excluded.  Use them for message passing
# while keeping the 6 semantic relations as supervision targets.
DEFAULT_SUPERVISION_RELATIONS: set[str] = {
    "IMPLEMENTS", "CONTAINS", "HAS_PARAM", "CALLS", "RELATED_TO", "REPLACES",
}

DEFAULT_MESSAGE_PASSING_RELATIONS: set[str] = DEFAULT_SUPERVISION_RELATIONS | {
    "EXPLAINS", "REFERENCES",
}


def empty_edge_index() -> torch.Tensor:
    return torch.empty((2, 0), dtype=torch.long)


def set_seed(seed: int) -> None:
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def reverse_edge_type(et: tuple[str, str, str]) -> tuple[str, str, str]:
    src, rel, dst = et
    return dst, f"rev_{rel}", src


def split_edge_index(
    edge_index: torch.Tensor,
    val_ratio: float,
    test_ratio: float,
    generator: torch.Generator,
) -> dict[str, torch.Tensor]:
    n = edge_index.size(1)
    if n == 0:
        return {"train": empty_edge_index(), "val": empty_edge_index(), "test": empty_edge_index()}
    perm    = torch.randperm(n, generator=generator)
    shuffled = edge_index[:, perm]
    if n < 3:
        return {"train": shuffled, "val": empty_edge_index(), "test": empty_edge_index()}

    val_count  = max(int(n * val_ratio),  int(val_ratio  > 0))
    test_count = max(int(n * test_ratio), int(test_ratio > 0 and n - val_count > 1))
    while n - val_count - test_count < 1:
        if test_count > 0:  test_count  -= 1
        elif val_count > 0: val_count   -= 1
        else: break

    train_count = n - val_count - test_count
    return {
        "train": shuffled[:, :train_count],
        "val":   shuffled[:, train_count: train_count + val_count],
        "test":  shuffled[:, train_count + val_count:],
    }


def sample_negative_edges(
    edge_index: torch.Tensor,
    num_src_nodes: int,
    num_dst_nodes: int,
    num_samples: int,
) -> torch.Tensor:
    if num_samples <= 0:
        return empty_edge_index()
    return negative_sampling(
        edge_index=edge_index,
        num_nodes=(num_src_nodes, num_dst_nodes),
        num_neg_samples=num_samples,
        method="sparse",
    )


# ── IMPROVEMENT 2: filter true positives from hard negatives ─────────────────
def sample_hard_negative_edges(
    edge_index: torch.Tensor,
    num_src_nodes: int,
    num_dst_nodes: int,
    num_samples: int,
) -> torch.Tensor:
    """
    Degree-distribution negative sampling (GraphSAGE-style) with true-positive
    filtering.  The previous version could accidentally sample real edges as
    negatives, injecting contradictory training signal.
    """
    if num_samples <= 0:
        return empty_edge_index()

    # Build positive set for fast lookup
    pos_set: set[tuple[int, int]] = set(
        zip(edge_index[0].tolist(), edge_index[1].tolist())
    )

    dst_indices = edge_index[1]
    degrees     = torch.bincount(dst_indices, minlength=num_dst_nodes).float()
    degrees    += 1.0
    weights     = degrees ** 0.75
    probs       = weights / weights.sum()

    # Over-sample then filter; retry up to 3 times to hit num_samples
    collected_src: list[int] = []
    collected_dst: list[int] = []
    oversample_factor = 3

    repeats    = (num_samples * oversample_factor // edge_index.size(1)) + 1
    src_pool   = edge_index[0].repeat(repeats)

    for _ in range(5):   # retry budget
        remaining = num_samples - len(collected_src)
        if remaining <= 0:
            break
        neg_dsts = torch.multinomial(probs, remaining * oversample_factor, replacement=True)
        srcs     = src_pool[:len(neg_dsts)]
        for s, d in zip(srcs.tolist(), neg_dsts.tolist()):
            if (s, d) not in pos_set:
                collected_src.append(s)
                collected_dst.append(d)
            if len(collected_src) >= num_samples:
                break

    if not collected_src:
        return empty_edge_index()

    src_t = torch.tensor(collected_src[:num_samples], dtype=torch.long)
    dst_t = torch.tensor(collected_dst[:num_samples], dtype=torch.long)
    return torch.stack([src_t, dst_t], dim=0)


def select_supervised_edge_types(
    data, allowed_relations: set[str], min_edges: int,
) -> list[tuple[str, str, str]]:
    selected = []
    for et in data.edge_types:
        _, relation, _ = et
        if relation.startswith("rev_"):
            continue
        num_edges = int(data[et].edge_index.size(1))
        if relation in allowed_relations and num_edges >= min_edges:
            selected.append(et)
    return sorted(selected)


def filter_graph_relations(data, allowed_base_relations: set[str] | None):
    if not allowed_base_relations:
        return data
    filtered  = data.clone()
    to_remove = [
        et for et in filtered.edge_types
        if (et[1][4:] if et[1].startswith("rev_") else et[1]) not in allowed_base_relations
    ]
    for et in to_remove:
        del filtered[et]
    return filtered


def minimum_supervision_edges_for_splits(val_ratio: float, test_ratio: float) -> int:
    return 3 if val_ratio > 0 or test_ratio > 0 else 1


def maybe_load_or_export_graph(
    graph_path, metadata_path, refresh_graph, uri, user, password,
):
    if graph_path.exists() and metadata_path.exists() and not refresh_graph:
        return load_graph_bundle(graph_path, metadata_path)
    return export_from_neo4j(
        uri=uri, user=user, password=password,
        graph_path=graph_path, metadata_path=metadata_path, add_reverse_edges=True,
    )


def clone_state_dict(model: torch.nn.Module) -> dict[str, torch.Tensor]:
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}


def build_split_state(
    data,
    supervised_edge_types: list[tuple[str, str, str]],
    val_ratio: float,
    test_ratio: float,
    seed: int,
) -> dict[tuple[str, str, str], dict[str, torch.Tensor]]:
    generator = torch.Generator().manual_seed(seed)
    split_state: dict[tuple[str, str, str], dict[str, torch.Tensor]] = {}
    for et in supervised_edge_types:
        ei           = data[et].edge_index.cpu()
        src_type, _, dst_type = et
        n_src = int(data[src_type].num_nodes)
        n_dst = int(data[dst_type].num_nodes)
        splits = split_edge_index(ei, val_ratio, test_ratio, generator)
        split_state[et] = {
            "full_pos":  ei,
            "train_pos": splits["train"],
            "val_pos":   splits["val"],
            "test_pos":  splits["test"],
            "val_neg":   sample_negative_edges(ei, n_src, n_dst, int(splits["val"].size(1))),
            "test_neg":  sample_negative_edges(ei, n_src, n_dst, int(splits["test"].size(1))),
        }
    return split_state


def build_train_graph(full_data, split_state) -> Any:
    """
    Removes val/test supervision edges from the training graph.
    NOTE: non-supervised message-passing edges (e.g. EXPLAINS, REFERENCES)
    are intentionally left intact – they provide structural context without
    leaking supervision labels.
    """
    train_data = full_data.clone()
    for et, state in split_state.items():
        train_edges = state["train_pos"].clone()
        train_data[et].edge_index = train_edges
        rev_et = reverse_edge_type(et)
        if rev_et in train_data.edge_types:
            train_data[rev_et].edge_index = train_edges.flip(0).contiguous()
    return train_data


# ── IMPROVEMENT 3: better early-stopping score ───────────────────────────────
def compute_early_stopping_score(
    val_metrics: dict[str, Any],
    split_state: dict[tuple[str, str, str], dict[str, torch.Tensor]],
    min_val_edges: int = 15,
) -> tuple[float, float]:
    """
    Use (val_macro_acc, val_ap) as the primary stopping signal instead of
    (val_min_acc, val_macro_acc, val_ap).

    val_min_acc is dominated by tiny splits (e.g. 1-2 validation edges for
    RELATED_TO / REPLACES) which are either 0 or 1 and provide no gradient
    of improvement for the first dozens of epochs.

    Relations with < min_val_edges positive validation samples are excluded
    from the stopping criterion.
    """
    per_type = val_metrics.get("per_type", {})
    macro_accs, aps = [], []
    for key, m in per_type.items():
        num_pos = m.get("num_pos", 0)
        if num_pos < min_val_edges:
            continue
        if _is_valid(m["accuracy"]):
            macro_accs.append(m["accuracy"])
        if _is_valid(m["average_precision"]):
            aps.append(m["average_precision"])

    macro_acc = sum(macro_accs) / len(macro_accs) if macro_accs else float("-inf")
    avg_ap    = sum(aps)        / len(aps)         if aps        else float("-inf")
    return macro_acc, avg_ap


def train_one_epoch(
    model: "HeteroLinkPredictor",
    optimizer: torch.optim.Optimizer,
    scheduler,           # may be None
    train_graph,
    split_state: dict[tuple[str, str, str], dict[str, torch.Tensor]],
    device: torch.device,
    negative_ratio: int = 4,
) -> tuple[float, dict[str, float]]:
    import math as _math
    model.train()
    optimizer.zero_grad()
    z_dict = model.encode(train_graph.x_dict, train_graph.edge_index_dict)

    losses, weights = [], []
    all_pos_scores, all_neg_scores = [], []

    for et, state in split_state.items():
        train_pos = state["train_pos"]
        if train_pos.numel() == 0:
            continue
        src_type, _, dst_type = et
        neg_ei = sample_negative_edges(
            state["full_pos"],
            int(train_graph[src_type].num_nodes),
            int(train_graph[dst_type].num_nodes),
            int(train_pos.size(1)) * max(1, int(negative_ratio)),
        )
        if neg_ei.numel() == 0:
            continue

        pos_scores = model.score_edges(z_dict, et, train_pos.to(device))
        neg_scores = model.score_edges(z_dict, et, neg_ei.to(device))
        logits = torch.cat([pos_scores, neg_scores])
        labels = torch.cat([torch.ones_like(pos_scores), torch.zeros_like(neg_scores)])
        bce    = F.binary_cross_entropy_with_logits(logits, labels)
        weight = _math.sqrt(train_pos.size(1))
        losses.append(bce * weight)
        weights.append(weight)
        all_pos_scores.append(pos_scores.detach().cpu())
        all_neg_scores.append(neg_scores.detach().cpu())

    if not losses:
        raise RuntimeError("No supervised edges available for training.")

    loss = sum(losses) / sum(weights)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
    optimizer.step()

    # IMPROVEMENT 4: step the scheduler if one was provided
    if scheduler is not None:
        # ReduceLROnPlateau needs the metric; cosine/step step on their own
        pass  # stepped in the training loop after val metrics are computed

    train_metrics = compute_classification_metrics(
        torch.cat(all_pos_scores), torch.cat(all_neg_scores), threshold=0.0,
    )
    return float(loss.item()), train_metrics


## Cell 5 — Embedding Export Helpers

**Improvements:** (1) L2 normalisation before export (`normalize=True`). (2) `build_export_tensors` for efficient `{node_type: Tensor}` storage. (3) Export on `full_data` (all relation types) by default. (4) Buffered `write_jsonl` instead of sequential per-row writes.

In [5]:
# Cell 5: Embedding export helpers (improved)

import io
import json
import re
from pathlib import Path

import torch
import torch.nn.functional as F
from neo4j import GraphDatabase


PROPERTY_NAME_PATTERN = re.compile(r"[A-Za-z_][A-Za-z0-9_]*")


def validate_property_name(name: str) -> None:
    if not PROPERTY_NAME_PATTERN.fullmatch(name):
        raise ValueError(
            "Neo4j property names may only contain letters, numbers, and "
            "underscores, and must not start with a number."
        )


@torch.no_grad()
def build_export_rows(
    model: "HeteroLinkPredictor",
    full_graph,
    metadata: dict,
    normalize: bool = True,
) -> list[dict]:
    """
    IMPROVEMENT 1 – L2 normalise before export.
    Graph-RAG retrieval uses cosine similarity; unnormalised vectors force
    the consumer to remember to normalise.  Doing it here is safer.

    IMPROVEMENT 2 – run inference on the graph passed in (full_data or
    working_data).  The caller should pass `full_data` if they want
    embeddings informed by ALL relation types (EXPLAINS, REFERENCES etc.)
    or `working_data` if they want the filtered view.  Documented clearly.
    """
    model.eval()
    z_dict = model.encode(full_graph.x_dict, full_graph.edge_index_dict)
    rows = []
    for node_type, embeddings in z_dict.items():
        if normalize:
            embeddings = F.normalize(embeddings, p=2, dim=-1)
        typed_meta = metadata["node_metadata"][node_type]
        if len(typed_meta) != embeddings.size(0):
            raise ValueError(
                f"Metadata mismatch for {node_type}: "
                f"{len(typed_meta)} rows vs {embeddings.size(0)} embeddings"
            )
        for idx, node_info in enumerate(typed_meta):
            rows.append({
                "element_id":   node_info["element_id"],
                "node_type":    node_type,
                "display_name": node_info["display_name"],
                "embedding":    embeddings[idx].cpu().tolist(),
            })
    return rows


@torch.no_grad()
def build_export_tensors(
    model: "HeteroLinkPredictor",
    full_graph,
    normalize: bool = True,
) -> dict[str, torch.Tensor]:
    """
    IMPROVEMENT 3 – efficient tensor-only export.
    Returns {node_type: Tensor(N, d)} which is 10-100× smaller than a list
    of dicts with Python float lists.  Use this for downstream torch code;
    use build_export_rows only for JSONL / Neo4j write-back.
    """
    model.eval()
    z_dict = model.encode(full_graph.x_dict, full_graph.edge_index_dict)
    if normalize:
        return {nt: F.normalize(z.cpu(), p=2, dim=-1) for nt, z in z_dict.items()}
    return {nt: z.cpu() for nt, z in z_dict.items()}


def write_embeddings_to_neo4j(
    rows: list[dict],
    uri: str,
    user: str,
    password: str,
    property_name: str,
    batch_size: int,
) -> None:
    validate_property_name(property_name)
    query = f"""
    UNWIND $rows AS row
    MATCH (n) WHERE elementId(n) = row.element_id
    SET n.{property_name} = row.embedding
    """
    driver = GraphDatabase.driver(uri, auth=(user, password))
    try:
        with driver.session() as session:
            for start in range(0, len(rows), batch_size):
                session.run(query, rows=rows[start: start + batch_size])
    finally:
        driver.close()


# ── IMPROVEMENT 4: buffered JSONL write ─────────────────────────────────────
def write_jsonl(rows: list[dict], path: Path) -> None:
    """Write rows to a JSONL file using a buffered writer for speed."""
    path.parent.mkdir(parents=True, exist_ok=True)
    buf = io.BytesIO()
    for row in rows:
        buf.write((json.dumps(row) + "\n").encode("utf-8"))
    path.write_bytes(buf.getvalue())


def load_model(checkpoint_path: Path, full_graph) -> "HeteroLinkPredictor":
    import warnings
    warnings.warn(
        "Loading model checkpoint with weights_only=False.",
        UserWarning, stacklevel=2,
    )
    cp     = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    cfg    = cp["model_config"]
    model  = HeteroLinkPredictor(
        metadata=full_graph.metadata(),
        scored_edge_types=cfg["scored_edge_types"],
        hidden_channels=cfg["hidden_channels"],
        out_channels=cfg["out_channels"],
        num_layers=cfg["num_layers"],
        dropout=cfg["dropout"],
        decoder_rank=cfg.get("decoder_rank", 32),
    )
    model.load_state_dict(cp["model_state"])
    return model


## Configuration

**Improvements:** (1) `refresh_graph=False`. (2) `min_supervision_edges=20` (excludes 4-edge and 7-edge relations from supervision). (3) `message_passing_relations` now includes EXPLAINS + REFERENCES. (4) `dropout=0.3`, `num_layers=3`, `negative_ratio=4`, `decoder_rank=32`.

In [6]:
# Cell 6: Configuration (improved)

CONFIG = {
    "uri":           DEFAULT_URI,
    "user":          DEFAULT_USER,
    "password":      DEFAULT_PASSWORD,
    "graph_path":    Path("outputs/hetero_graph.pt"),
    "metadata_path": Path("outputs/hetero_metadata.json"),
    "output_dir":    Path("outputs"),

    "epochs":   400,
    "patience":  40,

    # ── Model ──────────────────────────────────────────────
    "hidden_channels": 256,
    "out_channels":    256,
    "num_layers":      3,       # reduced from 4 → less over-smoothing on small graph
    "dropout":         0.3,     # raised from 0.1 → combats the overfitting seen in logs
    "decoder_rank":    32,      # low-rank bilinear (was full d² = 65 536 params/relation)

    # ── Optimiser ──────────────────────────────────────────
    "lr":           2e-4,
    "weight_decay": 1e-5,

    # ── Negative sampling ──────────────────────────────────
    "negative_ratio": 4,        # raised from 2 → harder negatives slow memorisation

    # ── Splits ─────────────────────────────────────────────
    "val_ratio":  0.1,
    "test_ratio": 0.1,
    "seed":       42,

    # IMPROVEMENT 1: raise min_supervision_edges so tiny relations
    # (RELATED_TO=4 edges, REPLACES=7 edges) are excluded from supervision.
    "min_supervision_edges": 20,

    # IMPROVEMENT 2: supervision only on the 6 semantic relations
    "supervision_relations": sorted(DEFAULT_SUPERVISION_RELATIONS),

    # IMPROVEMENT 3: message passing also uses the dense structural relations
    # (EXPLAINS=27 090 edges, REFERENCES=16 710 edges) for richer propagation.
    "message_passing_relations": sorted(DEFAULT_MESSAGE_PASSING_RELATIONS),

    # IMPROVEMENT 4: default to not re-fetching from Neo4j on every run
    "refresh_graph": True,

    # Min positive val edges for a relation to count in early-stopping score
    "min_val_edges_for_stopping": 15,
    # Min positive test edges for a relation to appear in headline test metrics
    "min_test_edges_for_reporting": 10,
}

CONFIG["output_dir"].mkdir(parents=True, exist_ok=True)
set_seed(CONFIG["seed"])
CONFIG


{'uri': 'bolt://localhost:7687',
 'user': 'neo4j',
 'password': 'pytorch2026',
 'graph_path': PosixPath('outputs/hetero_graph.pt'),
 'metadata_path': PosixPath('outputs/hetero_metadata.json'),
 'output_dir': PosixPath('outputs'),
 'epochs': 400,
 'patience': 40,
 'hidden_channels': 256,
 'out_channels': 256,
 'num_layers': 3,
 'dropout': 0.3,
 'decoder_rank': 32,
 'lr': 0.0002,
 'weight_decay': 1e-05,
 'negative_ratio': 4,
 'val_ratio': 0.1,
 'test_ratio': 0.1,
 'seed': 42,
 'min_supervision_edges': 20,
 'supervision_relations': ['CALLS',
  'CONTAINS',
  'HAS_PARAM',
  'IMPLEMENTS',
  'RELATED_TO',
  'REPLACES'],
 'message_passing_relations': ['CALLS',
  'CONTAINS',
  'EXPLAINS',
  'HAS_PARAM',
  'IMPLEMENTS',
  'REFERENCES',
  'RELATED_TO',
  'REPLACES'],
 'refresh_graph': True,
 'min_val_edges_for_stopping': 15,
 'min_test_edges_for_reporting': 10}

## Step 1: Export Neo4j Graph To PyG

In [7]:
full_data, metadata = maybe_load_or_export_graph(
    graph_path=CONFIG["graph_path"],
    metadata_path=CONFIG["metadata_path"],
    refresh_graph=CONFIG["refresh_graph"],
    uri=CONFIG["uri"],
    user=CONFIG["user"],
    password=CONFIG["password"],
)
print(full_data)
print(f"Feature dim: {metadata['feature_dim']}")
print(f"Node types:  {len(full_data.node_types)}")
print(f"Edge types:  {len(full_data.edge_types)}")


HeteroData(
  API_Class={ x=[142, 1024] },
  API_Endpoint={ x=[2747, 1024] },
  API_Function={ x=[264, 1024] },
  API_Method={ x=[271, 1024] },
  API_Parameter={ x=[1119, 1024] },
  CodeSnippet={ x=[1985, 1024] },
  Concept={ x=[17943, 1024] },
  DeprecatedAPI={ x=[7, 1024] },
  PyTorchConcept={ x=[7, 1024] },
  (API_Class, CONTAINS, API_Method)={ edge_index=[2, 271] },
  (API_Class, rev_CONTAINS, API_Endpoint)={ edge_index=[2, 142] },
  (API_Endpoint, CONTAINS, API_Class)={ edge_index=[2, 142] },
  (API_Endpoint, CONTAINS, API_Function)={ edge_index=[2, 224] },
  (API_Endpoint, EXPLAINS, Concept)={ edge_index=[2, 27090] },
  (API_Endpoint, IMPLEMENTS, CodeSnippet)={ edge_index=[2, 2343] },
  (API_Endpoint, REFERENCES, API_Endpoint)={ edge_index=[2, 16710] },
  (API_Endpoint, rev_REFERENCES, API_Endpoint)={ edge_index=[2, 16710] },
  (API_Function, HAS_PARAM, API_Parameter)={ edge_index=[2, 636] },
  (API_Function, RELATED_TO, PyTorchConcept)={ edge_index=[2, 4] },
  (API_Function, REP

## Step 2: Choose Supervision Relations And Build Splits

Note: `working_data` keeps the message-passing filter (all relations including EXPLAINS/REFERENCES). Supervision is drawn only from `supervision_relations`.

In [8]:
# Message passing uses ALL allowed relations (including EXPLAINS, REFERENCES)
working_data = filter_graph_relations(
    full_data,
    allowed_base_relations=set(CONFIG["message_passing_relations"]),
)

print("Message-passing edge types kept:")
for et in working_data.edge_types:
    print(f"   {et} -> {int(working_data[et].edge_index.size(1))} edges")

required_supervision_edges = max(
    CONFIG["min_supervision_edges"],
    minimum_supervision_edges_for_splits(CONFIG["val_ratio"], CONFIG["test_ratio"]),
)

supervised_edge_types = select_supervised_edge_types(
    working_data,
    allowed_relations=set(CONFIG["supervision_relations"]),
    min_edges=required_supervision_edges,
)

print(f"\nMinimum edges required per supervised relation: {required_supervision_edges}")
print("\nSupervised edge types:")
for et in supervised_edge_types:
    print(f"   {et} -> {int(working_data[et].edge_index.size(1))} edges")

if not supervised_edge_types:
    raise RuntimeError("No supervision edge types satisfied the filters.")

split_state = build_split_state(
    working_data,
    supervised_edge_types=supervised_edge_types,
    val_ratio=CONFIG["val_ratio"],
    test_ratio=CONFIG["test_ratio"],
    seed=CONFIG["seed"],
)

train_graph_cpu = build_train_graph(working_data, split_state)
train_graph_cpu


Message-passing edge types kept:
   ('API_Class', 'CONTAINS', 'API_Method') -> 271 edges
   ('API_Class', 'rev_CONTAINS', 'API_Endpoint') -> 142 edges
   ('API_Endpoint', 'CONTAINS', 'API_Class') -> 142 edges
   ('API_Endpoint', 'CONTAINS', 'API_Function') -> 224 edges
   ('API_Endpoint', 'EXPLAINS', 'Concept') -> 27090 edges
   ('API_Endpoint', 'IMPLEMENTS', 'CodeSnippet') -> 2343 edges
   ('API_Endpoint', 'REFERENCES', 'API_Endpoint') -> 16710 edges
   ('API_Endpoint', 'rev_REFERENCES', 'API_Endpoint') -> 16710 edges
   ('API_Function', 'HAS_PARAM', 'API_Parameter') -> 636 edges
   ('API_Function', 'RELATED_TO', 'PyTorchConcept') -> 4 edges
   ('API_Function', 'REPLACES', 'DeprecatedAPI') -> 7 edges
   ('API_Function', 'rev_CALLS', 'CodeSnippet') -> 48 edges
   ('API_Function', 'rev_CONTAINS', 'API_Endpoint') -> 224 edges
   ('API_Method', 'HAS_PARAM', 'API_Parameter') -> 483 edges
   ('API_Method', 'rev_CONTAINS', 'API_Class') -> 271 edges
   ('API_Parameter', 'rev_HAS_PARAM', 'API_

HeteroData(
  API_Class={ x=[142, 1024] },
  API_Endpoint={ x=[2747, 1024] },
  API_Function={ x=[264, 1024] },
  API_Method={ x=[271, 1024] },
  API_Parameter={ x=[1119, 1024] },
  CodeSnippet={ x=[1985, 1024] },
  Concept={ x=[17943, 1024] },
  DeprecatedAPI={ x=[7, 1024] },
  PyTorchConcept={ x=[7, 1024] },
  (API_Class, CONTAINS, API_Method)={ edge_index=[2, 217] },
  (API_Class, rev_CONTAINS, API_Endpoint)={ edge_index=[2, 114] },
  (API_Endpoint, CONTAINS, API_Class)={ edge_index=[2, 114] },
  (API_Endpoint, CONTAINS, API_Function)={ edge_index=[2, 180] },
  (API_Endpoint, EXPLAINS, Concept)={ edge_index=[2, 27090] },
  (API_Endpoint, IMPLEMENTS, CodeSnippet)={ edge_index=[2, 1875] },
  (API_Endpoint, REFERENCES, API_Endpoint)={ edge_index=[2, 16710] },
  (API_Endpoint, rev_REFERENCES, API_Endpoint)={ edge_index=[2, 16710] },
  (API_Function, HAS_PARAM, API_Parameter)={ edge_index=[2, 510] },
  (API_Function, RELATED_TO, PyTorchConcept)={ edge_index=[2, 4] },
  (API_Function, REP

## Step 3: Train The GNN

**Improvements active:** separate encoder/decoder weight decay, ReduceLROnPlateau scheduler, (macro_acc, avg_ap) early stopping over sufficient relations only.

In [9]:
# Cell 10: Training loop (improved)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
train_graph = train_graph_cpu.to(device)

model = HeteroLinkPredictor(
    metadata=train_graph.metadata(),
    scored_edge_types=supervised_edge_types,
    hidden_channels=CONFIG["hidden_channels"],
    out_channels=CONFIG["out_channels"],
    num_layers=CONFIG["num_layers"],
    dropout=CONFIG["dropout"],
    decoder_rank=CONFIG["decoder_rank"],
).to(device)

# IMPROVEMENT 1 – separate weight decay for decoder matrices.
# The encoder already benefits from weight_decay via Adam.  The bilinear
# U/V factors are prone to growing unbounded; applying a stronger decay
# stabilises training on rare-relation types.
encoder_params = list(model.encoder.parameters())
decoder_params = list(model.decoder.parameters())

optimizer = torch.optim.Adam([
    {"params": encoder_params, "weight_decay": CONFIG["weight_decay"]},
    {"params": decoder_params, "weight_decay": CONFIG["weight_decay"] * 10},
], lr=CONFIG["lr"])

# IMPROVEMENT 2 – ReduceLROnPlateau scheduler.
# The training log shows loss plateauing mid-run while val metrics stagnate.
# Halving the LR after 10 non-improving val epochs lets the model squeeze
# further without needing a manual schedule.
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",     # maximise val score
    factor=0.5,
    patience=10,
    min_lr=1e-5,
)

history: list[dict] = []
best_score = (float("-inf"), float("-inf"))
best_checkpoint = None
epochs_without_improvement = 0

for epoch in range(1, CONFIG["epochs"] + 1):
    train_loss, train_metrics = train_one_epoch(
        model=model,
        optimizer=optimizer,
        scheduler=None,          # stepped manually below after val metrics
        train_graph=train_graph,
        split_state=split_state,
        device=device,
        negative_ratio=CONFIG["negative_ratio"],
    )

    val_thresholds, tuned_val_metrics = tune_thresholds_from_split(
        model=model,
        message_graph=train_graph,
        split_state=split_state,
        split_name="val",
        device=device,
    )

    # IMPROVEMENT 3 – use (macro_acc, avg_ap) over sufficient relations only.
    # The old (min_acc, macro_acc, ap) tuple was dominated by 1-edge relations
    # that give 0/1 signals, firing early stopping arbitrarily early.
    val_macro_acc, val_avg_ap = compute_early_stopping_score(
        tuned_val_metrics,
        split_state,
        min_val_edges=CONFIG["min_val_edges_for_stopping"],
    )
    val_score = (val_macro_acc, val_avg_ap)

    # Step scheduler on the primary metric
    scheduler.step(val_macro_acc)

    epoch_record = {
        "epoch":        epoch,
        "train_loss":   train_loss,
        "train_metrics": train_metrics,
        "val_thresholds": val_thresholds,
        "val_metrics":  tuned_val_metrics,
        "lr":           optimizer.param_groups[0]["lr"],
    }
    history.append(epoch_record)

    if val_score > best_score:
        best_score = val_score
        epochs_without_improvement = 0
        best_checkpoint = {
            "model_state": clone_state_dict(model),
            "model_config": {
                "hidden_channels": CONFIG["hidden_channels"],
                "out_channels":    CONFIG["out_channels"],
                "num_layers":      CONFIG["num_layers"],
                "dropout":         CONFIG["dropout"],
                "decoder_rank":    CONFIG["decoder_rank"],
                "scored_edge_types": supervised_edge_types,
            },
            "thresholds":       val_thresholds,
            "best_val_metrics": tuned_val_metrics,
        }
        torch.save(best_checkpoint, CONFIG["output_dir"] / "best_model.pt")
    else:
        epochs_without_improvement += 1

    print(
        f"Epoch {epoch:03d} | loss={train_loss:.4f} | "
        f"train_acc={train_metrics['accuracy']:.4f} | "
        f"val_macro_acc={val_macro_acc:.4f} | val_ap={val_avg_ap:.4f} | "
        f"lr={optimizer.param_groups[0]['lr']:.2e}"
    )

    if epochs_without_improvement >= CONFIG["patience"]:
        print(f"Early stopping after {epoch} epochs.")
        break

if best_checkpoint is None:
    raise RuntimeError("Training finished without a valid checkpoint.")

torch.save(split_state,      CONFIG["output_dir"] / "split_state.pt")
torch.save(train_graph_cpu,  CONFIG["output_dir"] / "train_graph.pt")

print("Saved best checkpoint →", (CONFIG["output_dir"] / "best_model.pt").resolve())
print("Best val score (macro_acc, avg_ap):", best_score)


Epoch 001 | loss=1.3787 | train_acc=0.5600 | val_macro_acc=0.6183 | val_ap=0.5983 | lr=2.00e-04
Epoch 002 | loss=1.4306 | train_acc=0.7895 | val_macro_acc=0.6263 | val_ap=0.6022 | lr=2.00e-04
Epoch 003 | loss=1.0537 | train_acc=0.6642 | val_macro_acc=0.6575 | val_ap=0.6375 | lr=2.00e-04
Epoch 004 | loss=0.9917 | train_acc=0.6669 | val_macro_acc=0.6926 | val_ap=0.6629 | lr=2.00e-04
Epoch 005 | loss=0.8430 | train_acc=0.7145 | val_macro_acc=0.7131 | val_ap=0.6730 | lr=2.00e-04
Epoch 006 | loss=0.7514 | train_acc=0.7251 | val_macro_acc=0.7159 | val_ap=0.6693 | lr=2.00e-04
Epoch 007 | loss=0.7030 | train_acc=0.7458 | val_macro_acc=0.7280 | val_ap=0.6773 | lr=2.00e-04
Epoch 008 | loss=0.6601 | train_acc=0.7526 | val_macro_acc=0.7259 | val_ap=0.6798 | lr=2.00e-04
Epoch 009 | loss=0.6731 | train_acc=0.7332 | val_macro_acc=0.7442 | val_ap=0.6839 | lr=2.00e-04
Epoch 010 | loss=0.6161 | train_acc=0.7576 | val_macro_acc=0.7529 | val_ap=0.6803 | lr=2.00e-04
Epoch 011 | loss=0.5398 | train_acc=0.78

## Step 4: Evaluate Best Model On Test Edges

**Improvements active:** val-tuned thresholds applied to test (never re-tuned on test), `insufficient_data` flag on tiny splits, MRR + Hits@k reported.

In [10]:
# Cell 11: Evaluate best model on test edges (improved)

best_model = HeteroLinkPredictor(
    metadata=train_graph.metadata(),
    scored_edge_types=supervised_edge_types,
    hidden_channels=CONFIG["hidden_channels"],
    out_channels=CONFIG["out_channels"],
    num_layers=CONFIG["num_layers"],
    dropout=CONFIG["dropout"],
    decoder_rank=CONFIG["decoder_rank"],
).to(device)
best_model.load_state_dict(best_checkpoint["model_state"])

# Default-threshold test (val thresholds → test set, never tuned on test data)
test_metrics = evaluate_model(
    model=best_model,
    message_graph=train_graph,
    split_state=split_state,
    split_name="test",
    device=device,
    thresholds=best_checkpoint["thresholds"],     # tuned on VAL, applied to TEST
    min_edges_for_reporting=CONFIG["min_test_edges_for_reporting"],
)

# Highlight which relations had insufficient test data
print("=== Test Metrics (val-tuned thresholds) ===")
print(json.dumps(test_metrics["overall"], indent=2))

print("\n=== Per-relation breakdown ===")
for rel, m in test_metrics["per_type"].items():
    flag = " ⚠ insufficient data" if m.get("insufficient_data") else ""
    print(
        f"  {rel}{flag}\n"
        f"    auc={m['roc_auc']:.3f}  ap={m['average_precision']:.3f}  "
        f"acc={m['accuracy']:.3f}  mrr={m['mrr']:.3f}  "
        f"h@1={m['hits@1']:.3f}  h@10={m['hits@10']:.3f}  "
        f"(n_pos={m['num_pos']})\n"
    )

summary = {
    "device": str(device),
    "supervised_edge_types":    [list(et) for et in supervised_edge_types],
    "message_passing_relations": sorted(CONFIG["message_passing_relations"]),
    "best_thresholds":          best_checkpoint["thresholds"],
    "best_val_metrics":         best_checkpoint["best_val_metrics"],
    "test_metrics":             test_metrics,
    "history":                  history,
}
(CONFIG["output_dir"] / "training_summary.json").write_text(
    json.dumps(summary, indent=2), encoding="utf-8"
)


=== Test Metrics (val-tuned thresholds) ===
{
  "roc_auc": 0.6561940333678952,
  "average_precision": 0.6596058965886681,
  "accuracy": 0.6735436923584892,
  "threshold": 0.0,
  "mrr": 0.10534514486789703,
  "hits@1": 0.07038834691047668,
  "hits@5": 0.13349515199661255,
  "hits@10": 0.14077669382095337,
  "macro_accuracy": 0.7921796639760336,
  "min_accuracy": 0.5940170884132385,
  "num_relations": 6
}

=== Per-relation breakdown ===
  API_Class__CONTAINS__API_Method
    auc=0.923  ap=0.922  acc=0.870  mrr=0.570  h@1=0.370  h@10=0.963  (n_pos=27)

  API_Endpoint__CONTAINS__API_Class
    auc=0.939  ap=0.933  acc=0.893  mrr=0.690  h@1=0.500  h@10=1.000  (n_pos=14)

  API_Endpoint__CONTAINS__API_Function
    auc=1.000  ap=1.000  acc=1.000  mrr=1.000  h@1=1.000  h@10=1.000  (n_pos=22)

  API_Endpoint__IMPLEMENTS__CodeSnippet
    auc=0.588  ap=0.600  acc=0.594  mrr=0.066  h@1=0.034  h@10=0.090  (n_pos=234)

  API_Function__HAS_PARAM__API_Parameter
    auc=0.546  ap=0.475  acc=0.667  mrr=0.

409283

## Step 5: Export Final Graph-Aware Embeddings

**Improvements active:** inference on `full_data` (all relations), L2 normalisation, efficient tensor export, buffered JSONL write.

In [11]:
# Cell 12: Export final graph-aware embeddings (improved)

best_model_cpu = best_model.cpu()

# IMPROVEMENT 1 – use full_data so EXPLAINS/REFERENCES edges inform embeddings
# IMPROVEMENT 2 – L2 normalise for cosine-sim retrieval
# IMPROVEMENT 3 – save efficient tensor dict in addition to JSONL rows

full_data_cpu = full_data.cpu()     # full graph with ALL relation types

# Efficient tensor export (for torch downstream consumers)
z_tensors = build_export_tensors(best_model_cpu, full_data_cpu, normalize=True)
pt_path = CONFIG["output_dir"] / "gnn_embeddings.pt"
torch.save(z_tensors, pt_path)

# JSONL export (for Neo4j write-back or inspection)
rows       = build_export_rows(best_model_cpu, full_data_cpu, metadata, normalize=True)
jsonl_path = CONFIG["output_dir"] / "gnn_embeddings.jsonl"
write_jsonl(rows, jsonl_path)       # IMPROVEMENT 4: buffered write

print(f"Saved tensor embeddings → {pt_path.resolve()}")
print(f"Saved JSONL embeddings  → {jsonl_path.resolve()}")
print(f"Total nodes exported: {sum(t.size(0) for t in z_tensors.values())}")
print("Sample z-norm (should be ~1.0):",
      float(z_tensors[list(z_tensors)[0]][0].norm().item()))
rows[0]


Saved tensor embeddings → /home/rahul/code/tdl_proj/Structural-Coder/outputs/gnn_embeddings.pt
Saved JSONL embeddings  → /home/rahul/code/tdl_proj/Structural-Coder/outputs/gnn_embeddings.jsonl
Total nodes exported: 24485
Sample z-norm (should be ~1.0): 0.9999999403953552


{'element_id': '4:170763f8-8d9f-40c2-b8d0-f5cb2fbaa0a1:22715',
 'node_type': 'API_Class',
 'display_name': 'torch.SymInt',
 'embedding': [0.06825190037488937,
  -0.017661945894360542,
  0.0500919371843338,
  0.092707060277462,
  -0.05495302379131317,
  -0.010747471824288368,
  0.051408737897872925,
  -0.04815876856446266,
  0.04730938374996185,
  0.08685020357370377,
  -0.02193165011703968,
  0.026051117107272148,
  -0.056083377450704575,
  -0.029671194031834602,
  0.0012173515278846025,
  -0.05635058507323265,
  0.046112190932035446,
  -0.008298717439174652,
  0.19734574854373932,
  -0.017027854919433594,
  -0.09881916642189026,
  0.008639168925583363,
  0.04644425958395004,
  -0.005529773887246847,
  -0.05833331122994423,
  -0.08637715131044388,
  -0.025544943287968636,
  -0.02056991495192051,
  0.07371015846729279,
  -0.0807209312915802,
  -0.031053317710757256,
  -0.10202738642692566,
  0.043175842612981796,
  0.11581824719905853,
  -0.05404556170105934,
  0.11420808732509613,
  0.

## Optional: Write Back To Neo4j

Uncomment only if you want the trained GNN vectors stored back in Neo4j.

In [12]:
# write_embeddings_to_neo4j(
#     rows=rows,
#     uri=CONFIG["uri"],
#     user=CONFIG["user"],
#     password=CONFIG["password"],
#     property_name="gnn_embedding",
#     batch_size=256,
# )
# print("Wrote gnn_embedding back to Neo4j")
